In [ ]:
"""Deterministic, standard-library-only corpus construction.

Authoritative protocol: plans/260915-0955-visual-delta-fusion-pilot/plan.md
and phase-01-local-module.md. This module never imports torch, transformers,
numpy, or PIL; caption/paraphrase generation backends live in caption.py and
are loaded lazily there. Everything here is pure functions over plain Python
data so it is testable without a GPU and reproducible on Kaggle.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Protocol, Sequence

UNAVAILABLE = "unavailable"
TITLE_PREFIX = "Title: "
CUE_SEPARATOR = "; Visual cues: "
ARMS: tuple[str, ...] = ("title-only", "null", "real", "shuffle", "paraphrase")
CUE_CAPS: tuple[int, ...] = (16, 32, 64)
PRIMARY_CAP = 32
BIN_SIZE = 64
MAX_CSFT_TOKENS = 1024

_WHITESPACE_RE = re.compile(r"\s+")


def normalize_whitespace(text: str) -> str:
    """Collapse runs of whitespace and strip the ends. Never touches wording."""
    return _WHITESPACE_RE.sub(" ", text).strip()


def fuse_text(title: str, cue: str) -> str:
    """Build the fused item text ``Title: T; Visual cues: Z`` (plan.md formula).

    ``title`` is used verbatim (callers decide normalization); ``cue`` falls
    back to the literal ``unavailable`` marker when empty, never to an
    invented placeholder.
    """
    if not title:
        raise ValueError("fuse_text requires a non-empty title")
    resolved_cue = cue if cue else UNAVAILABLE
    return f"{TITLE_PREFIX}{title}{CUE_SEPARATOR}{resolved_cue}"


@dataclass(frozen=True)
class CatalogRow:
    """One catalog item with everything downstream stages need, in one place.

    ``order_index`` preserves the source's original row order (title-source
    order for a single domain, or occurrence order across the mixed corpus);
    string keys elsewhere never imply this ordering.
    """

    domain: str
    source_item_id: str
    downstream_item_id: int | None
    parent_asin: str | None
    source_title: str
    order_index: int
    image_status: str = "unknown"
    caption_status: str = "unknown"
    raw_caption: str | None = None


def build_catalog_rows(
    item_titles: Mapping[str, str],
    item_asin_map: Mapping[str, Mapping[str, object]] | None = None,
    domain: str = "",
) -> list[CatalogRow]:
    """Join a downstream ``item_titles`` map with an optional ASIN map.

    ``item_titles`` keys are string downstream IDs, contract-compatible with
    the committed ``item_titles.json`` (IDs ``1..N``, no gaps). Row order
    follows ascending numeric ID, matching every existing item_texts.txt
    convention in this repository. Every title row is kept even when no ASIN
    entry exists for it (recorded, not dropped) so coverage can be reported
    honestly rather than by silent omission.
    """
    if not item_titles:
        raise ValueError("item_titles must not be empty")
    try:
        ordered_ids = sorted(item_titles, key=int)
    except ValueError as exc:
        raise ValueError("item_titles keys must be numeric downstream IDs") from exc
    numeric_ids = [int(item_id) for item_id in ordered_ids]
    expected = list(range(1, max(numeric_ids) + 1))
    if numeric_ids != expected:
        raise ValueError(
            f"item_titles for domain {domain!r} must cover exactly 1..N with no gaps; "
            f"got {len(numeric_ids)} rows spanning {numeric_ids[0]}..{numeric_ids[-1]}"
        )
    asin_by_downstream: dict[str, Mapping[str, object]] = {}
    if item_asin_map is not None:
        items = item_asin_map.get("items", item_asin_map)
        if not isinstance(items, Mapping):
            raise ValueError("item_asin_map must contain an 'items' mapping or be one itself")
        title_ids = set(ordered_ids)
        for raw_id, record in items.items():
            downstream_id = str(int(raw_id))
            if downstream_id not in title_ids:
                raise ValueError(
                    f"item_asin_map contains downstream ID {downstream_id} absent from item_titles"
                )
            if downstream_id in asin_by_downstream:
                raise ValueError(f"duplicate downstream ID {downstream_id} in item_asin_map for {domain!r}")
            if not isinstance(record, Mapping):
                raise ValueError(f"item_asin_map record {downstream_id} is not an object")
            mapped_title = record.get("title")
            if mapped_title is not None and str(mapped_title) != str(item_titles[downstream_id]):
                raise ValueError(
                    f"title mismatch for downstream ID {downstream_id}: "
                    f"item_titles={item_titles[downstream_id]!r}, map={mapped_title!r}"
                )
            if not record.get("parent_asin"):
                raise ValueError(f"item_asin_map record {downstream_id} has no parent_asin")
            asin_by_downstream[downstream_id] = record
    rows: list[CatalogRow] = []
    for order_index, item_id in enumerate(ordered_ids):
        record = asin_by_downstream.get(item_id)
        rows.append(
            CatalogRow(
                domain=domain,
                source_item_id=item_id,
                downstream_item_id=int(item_id),
                parent_asin=str(record["parent_asin"]) if record else None,
                source_title=item_titles[item_id],
                order_index=order_index,
                image_status="unmapped" if record is None else "pending",
            )
        )
    return rows


def compute_training_frequencies(train_sequences: Iterable[Sequence[int]]) -> dict[int, int]:
    """Count item occurrences across training-only interaction sequences.

    Callers must pass training-split sequences only (plan.md: "Frequency and
    strata use training interactions only"); this function has no way to
    enforce that boundary itself and trusts the caller's split.
    """
    frequencies: dict[int, int] = {}
    for sequence in train_sequences:
        for item_id in sequence:
            frequencies[item_id] = frequencies.get(item_id, 0) + 1
    return frequencies


def frequency_bin_derangement(
    available_ids: Sequence[int],
    frequencies: Mapping[int, float],
    seed: int,
    bin_size: int = BIN_SIZE,
) -> dict[int, int]:
    """Return a donor map ``item_id -> donor_item_id`` with no fixed points.

    Mirrors the frequency-bin-then-roll-by-one convention already accepted in
    this repository (``baby_sealed_manifest.py:build_derangement_for_seed``),
    reimplemented with ``random.Random`` since this workstation and this
    corpus family carry no numpy dependency. Items outside ``available_ids``
    are never donors and never receive a mapping; callers treat them as
    ``unavailable`` in every arm that consults this map.
    """
    unique_ids = list(dict.fromkeys(available_ids))
    if len(unique_ids) < 2:
        raise RuntimeError(f"derangement for seed {seed} needs at least two available items")
    ordered = sorted(unique_ids, key=lambda item_id: (frequencies.get(item_id, 0.0), item_id))
    bins = [ordered[start : start + bin_size] for start in range(0, len(ordered), bin_size)]
    if len(bins) > 1 and len(bins[-1]) < 2:
        bins[-2] = bins[-2] + bins[-1]
        bins.pop()
    rng = random.Random(seed)
    mapping: dict[int, int] = {}
    for bin_ids in bins:
        shuffled = list(bin_ids)
        rng.shuffle(shuffled)
        rolled = shuffled[1:] + shuffled[:1]
        for original, donor in zip(shuffled, rolled):
            mapping[original] = donor
    fixed_points = [item_id for item_id in unique_ids if mapping[item_id] == item_id]
    if fixed_points:
        raise RuntimeError(f"derangement for seed {seed} has fixed points: {fixed_points}")
    if sorted(mapping.values()) != sorted(mapping.keys()):
        raise RuntimeError(f"derangement for seed {seed} is not a permutation")
    return mapping


class Tokenizer(Protocol):
    """Minimal tokenizer boundary so token budgeting is testable without a GPU."""

    def encode(self, text: str) -> list[int]: ...


def truncate_to_token_cap(text: str, cap: int, tokenizer: Tokenizer) -> tuple[str, int]:
    """Truncate ``text`` to at most ``cap`` tokens at a whole-word boundary.

    Binary search over word count, assuming token count is non-decreasing in
    word count for the supplied tokenizer (true for word-piece/BPE
    tokenizers on additive text). Returns ``("", 0)`` for empty input or when
    even the first word already exceeds ``cap`` tokens.
    """
    words = text.split()
    if not words or cap <= 0:
        return "", 0
    low, high = 0, len(words)
    best_text, best_tokens = "", 0
    while low <= high:
        mid = (low + high) // 2
        candidate = " ".join(words[:mid]) if mid else ""
        token_count = len(tokenizer.encode(candidate)) if candidate else 0
        if token_count <= cap:
            best_text, best_tokens = candidate, token_count
            low = mid + 1
        else:
            high = mid - 1
    return best_text, best_tokens


@dataclass(frozen=True)
class ArmTexts:
    """The five arm strings for a single catalog item, plus donor provenance."""

    downstream_item_id: int
    texts: dict[str, str]
    shuffle_donor_id: int | None


def build_arms(
    rows: Sequence[CatalogRow],
    real_cues: Mapping[int, str],
    paraphrase_cues: Mapping[int, str],
    shuffle_donor_map: Mapping[int, int],
) -> list[ArmTexts]:
    """Build the five arms with one shared cue-availability mask."""
    if set(real_cues) != set(paraphrase_cues):
        raise ValueError("real and paraphrase availability masks must be identical")
    available_ids = {item_id for item_id, cue in real_cues.items() if cue}
    if set(real_cues) != available_ids:
        raise ValueError("real_cues must omit unavailable items rather than contain empty cues")
    if set(shuffle_donor_map) != available_ids or set(shuffle_donor_map.values()) != available_ids:
        raise ValueError("shuffle donor map must be a permutation of exactly the available IDs")
    if any(item_id == donor_id for item_id, donor_id in shuffle_donor_map.items()):
        raise ValueError("shuffle donor map contains a fixed point")
    arms: list[ArmTexts] = []
    for row in rows:
        item_id = row.downstream_item_id
        if item_id is None:
            raise ValueError(f"row {row.source_item_id!r} has no downstream_item_id")
        donor_id = shuffle_donor_map.get(item_id)
        real_cue = real_cues.get(item_id)
        shuffle_cue = real_cues.get(donor_id) if donor_id is not None else None
        paraphrase_cue = paraphrase_cues.get(item_id)
        texts = {
            "title-only": row.source_title,
            "null": fuse_text(row.source_title, UNAVAILABLE),
            "real": fuse_text(row.source_title, real_cue or UNAVAILABLE),
            "shuffle": fuse_text(row.source_title, shuffle_cue or UNAVAILABLE),
            "paraphrase": fuse_text(row.source_title, paraphrase_cue or UNAVAILABLE),
        }
        arms.append(ArmTexts(item_id, texts, donor_id))
    _assert_arm_consistency(rows, arms)
    return arms


def _assert_arm_consistency(rows: Sequence[CatalogRow], arms: Sequence[ArmTexts]) -> None:
    if len(rows) != len(arms):
        raise RuntimeError("arm count must equal row count")
    for row, arm in zip(rows, arms):
        if row.downstream_item_id != arm.downstream_item_id:
            raise RuntimeError("arm ordering drifted from row ordering")
        for name in ARMS:
            if name not in arm.texts:
                raise RuntimeError(f"item {arm.downstream_item_id} is missing arm {name!r}")
        title_segment = f"{TITLE_PREFIX}{row.source_title}"
        for name in ("null", "real", "shuffle", "paraphrase"):
            if not arm.texts[name].startswith(title_segment):
                raise RuntimeError(
                    f"item {arm.downstream_item_id} arm {name!r} does not share the title segment"
                )


def compute_common_history_suffix(
    history_ids: Sequence[int],
    target_title: str,
    arm_text_by_id: Mapping[int, str],
    tokenizer: Tokenizer,
    special_token_overhead: int = 8,
    max_tokens: int = MAX_CSFT_TOKENS,
) -> tuple[list[int], int]:
    """Keep the longest whole-item suffix of the rendered CSFT sequence."""
    if max_tokens < 0 or special_token_overhead < 0:
        raise ValueError("token budget and overhead must be non-negative")
    if len(tokenizer.encode(target_title)) + special_token_overhead > max_tokens:
        raise RuntimeError("target title plus special tokens already exceed max_tokens")
    for item_id in history_ids:
        if item_id not in arm_text_by_id:
            raise KeyError(f"history item {item_id} has no arm text")
    for start in range(len(history_ids) + 1):
        retained = list(history_ids[start:])
        rendered = " ".join([*(arm_text_by_id[item_id] for item_id in retained), target_title])
        used = len(tokenizer.encode(rendered)) + special_token_overhead
        if used <= max_tokens:
            return retained, used
    raise AssertionError("empty history must fit after target preflight")


def compute_shared_history_suffix(
    history_ids: Sequence[int],
    target_title: str,
    arm_text_by_id_variants: Mapping[str, Mapping[int, str]],
    tokenizer: Tokenizer,
    special_token_overhead: int = 8,
    max_tokens: int = MAX_CSFT_TOKENS,
) -> tuple[list[int], dict[str, int]]:
    """Choose one suffix valid for every registered arm/cap variant."""
    if not arm_text_by_id_variants:
        raise ValueError("at least one arm variant is required")
    candidates = [
        compute_common_history_suffix(
            history_ids, target_title, texts, tokenizer, special_token_overhead, max_tokens
        )[0]
        for texts in arm_text_by_id_variants.values()
    ]
    keep = min(map(len, candidates))
    suffix = list(history_ids[-keep:]) if keep else []
    budgets: dict[str, int] = {}
    for name, texts in arm_text_by_id_variants.items():
        rendered = " ".join([*(texts[item_id] for item_id in suffix), target_title])
        budgets[name] = len(tokenizer.encode(rendered)) + special_token_overhead
        if budgets[name] > max_tokens:
            raise RuntimeError(f"shared suffix exceeds budget for variant {name!r}")
    return suffix, budgets


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()
def atomic_write_text(path: Path, content: str) -> str:
    """Write UTF-8 content atomically and return its digest."""
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = content.encode("utf-8")
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_bytes(payload)
    os.replace(tmp_path, path)
    return sha256_bytes(payload)

def write_arm_corpora(arms: Sequence[ArmTexts], out_dir: Path) -> dict[str, dict[str, str]]:
    """Write ``item_texts.{json,txt}`` per arm, atomically, plus a hash ledger.

    ``item_texts.json`` uses string keys (matching every existing
    ``item_titles.json``-shaped file in this repository); ``item_texts.txt``
    has one line per item in ascending downstream-ID order, mirroring
    ``info/item_titles.txt``. A source title or cue containing a literal
    newline would otherwise silently split the TXT corpus, so any such text
    is rejected here rather than laundered.
    """
    if not arms:
        raise ValueError("write_arm_corpora requires at least one item")
    ordered = sorted(arms, key=lambda arm: arm.downstream_item_id)
    hashes: dict[str, dict[str, str]] = {}
    for arm_name in ARMS:
        json_map: dict[str, str] = {}
        lines: list[str] = []
        for arm in ordered:
            text = arm.texts[arm_name]
            if "\n" in text or "\r" in text:
                raise ValueError(
                    f"item {arm.downstream_item_id} arm {arm_name!r} contains a newline; "
                    "the flat .txt corpus cannot represent it without splitting rows"
                )
            json_map[str(arm.downstream_item_id)] = text
            lines.append(text)
        arm_dir = out_dir / arm_name
        json_hash = atomic_write_text(arm_dir / "item_texts.json", json.dumps(json_map, sort_keys=True))
        txt_hash = atomic_write_text(arm_dir / "item_texts.txt", "\n".join(lines) + "\n")
        hashes[arm_name] = {"item_texts.json": json_hash, "item_texts.txt": txt_hash}
    return hashes


SHARD_SIZE = 512


def _canonical_json(value: object) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def shard_records(records: Sequence[Mapping[str, object]], shard_size: int = SHARD_SIZE) -> list[list[Mapping[str, object]]]:
    """Partition records without reordering or dropping rows."""
    if shard_size <= 0:
        raise ValueError("shard_size must be positive")
    return [list(records[start : start + shard_size]) for start in range(0, len(records), shard_size)]


def write_sharded_records(
    records: Sequence[Mapping[str, object]],
    out_dir: Path,
    identity: Mapping[str, object],
    shard_size: int = SHARD_SIZE,
) -> dict[str, object]:
    """Write completed JSONL shards and an atomic, hash-pinned manifest.

    A shard is visible in the manifest only after its temporary JSONL file has
    been atomically renamed and its digest computed. Existing output is never
    reused implicitly: its identity must exactly equal ``identity``.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "shard-manifest.json"
    if manifest_path.is_file():
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        if existing.get("identity") != dict(identity):
            raise RuntimeError("existing shard manifest identity mismatch; refusing resume")
    shards: list[dict[str, object]] = []
    for index, chunk in enumerate(shard_records(records, shard_size)):
        payload = "".join(_canonical_json(record) + "\n" for record in chunk).encode("utf-8")
        shard_path = out_dir / f"shard-{index:05d}.jsonl"
        temp_path = shard_path.with_suffix(".jsonl.partial")
        temp_path.write_bytes(payload)
        os.replace(temp_path, shard_path)
        shards.append({
            "index": index,
            "path": shard_path.name,
            "records": len(chunk),
            "sha256": sha256_bytes(payload),
            "completion_status": "complete",
        })
    manifest = {
        "identity": dict(identity),
        "shard_size": shard_size,
        "record_count": len(records),
        "shard_count": len(shards),
        "shards": shards,
        "completion_status": "complete",
    }
    atomic_write_text(manifest_path, _canonical_json(manifest) + "\n")
    return manifest


def validate_shard_manifest(manifest_path: Path, expected_identity: Mapping[str, object]) -> dict[str, object]:
    """Validate every shard's identity, completion marker and SHA256."""
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("identity") != dict(expected_identity):
        raise RuntimeError("shard manifest identity mismatch")
    if manifest.get("completion_status") != "complete":
        raise RuntimeError("shard manifest is not complete")
    if manifest.get("shard_count") != len(manifest.get("shards", [])):
        raise RuntimeError("shard manifest count mismatch")
    for shard in manifest["shards"]:
        if shard.get("completion_status") != "complete":
            raise RuntimeError(f"shard {shard.get('index')} is incomplete")
        path = manifest_path.parent / str(shard["path"])
        if not path.is_file():
            raise RuntimeError(f"shard file missing: {path}")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest != shard.get("sha256"):
            raise RuntimeError(f"shard hash mismatch: {path}")
    return manifest


def load_shard_records(
    out_dir: Path, expected_identity: Mapping[str, object] | None = None
) -> list[dict[str, object]]:
    """Read a validated shard set back into one flat, catalog-ordered list.

    Complements `write_sharded_records`: shards are always written in
    catalog order and never reordered, so concatenating each shard's JSONL
    lines in the manifest's own shard order reconstructs the original record
    order exactly. Used to resume an interrupted generation run from its
    last checkpoint instead of restarting from item 0.

    A real cross-push resume hit `JSONDecodeError` reproducibly on a shard
    whose bytes matched the manifest hash and parsed as 512 valid physical
    JSONL lines. The shard contains U+0085 (Unicode NEL) inside one JSON
    string. `str.splitlines()` treats NEL as a line boundary, so decoding the
    whole payload before splitting manufactured a 513th line and truncated
    that record even though the file was valid.

    JSONL is byte-delimited by LF because `write_sharded_records` emits
    exactly `b"\\n"` between records. Split the validated payload on that
    delimiter before decoding each record; Unicode line-separator characters
    inside JSON strings then remain data. Hashing and parsing still share the
    same single byte read.
    """
    manifest_path = out_dir / "shard-manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if expected_identity is not None:
        if manifest.get("identity") != dict(expected_identity):
            raise RuntimeError("shard manifest identity mismatch")
        if manifest.get("completion_status") != "complete":
            raise RuntimeError("shard manifest is not complete")
        if manifest.get("shard_count") != len(manifest.get("shards", [])):
            raise RuntimeError("shard manifest count mismatch")
    records: list[dict[str, object]] = []
    for shard in manifest["shards"]:
        shard_path = out_dir / str(shard["path"])
        last_error: Exception | None = None
        for attempt in range(3):
            try:
                payload = shard_path.read_bytes()
                if expected_identity is not None:
                    if shard.get("completion_status") != "complete":
                        raise RuntimeError(f"shard {shard.get('index')} is incomplete")
                    digest = sha256_bytes(payload)
                    if digest != shard.get("sha256"):
                        raise RuntimeError(f"shard hash mismatch: {shard_path}")
                shard_records = [
                    json.loads(line.decode("utf-8")) for line in payload.split(b"\n") if line
                ]
                break
            except (json.JSONDecodeError, RuntimeError, UnicodeDecodeError) as exc:
                last_error = exc
                if attempt < 2:
                    time.sleep(0.5 * (attempt + 1))
        else:
            raise RuntimeError(
                f"shard file {shard_path} failed to read/verify/parse after 3 attempts: "
                f"{last_error}"
            ) from last_error
        records.extend(shard_records)
    return records


In [ ]:
"""AmazonMix-6 six-domain crosswalk with paper-verified block structure.

The mixed `AmazonMix-6` catalog is one global item list of 108,753 rows. Its
six domains and boundaries are established by the LLM2Rec paper, Table 1
(KDD'25): Arts (12,454), Electronics (20,150), Home_and_Kitchen (33,478),
Video_Games (9,517), Movies_and_TV (13,190), Tools_and_Home_Improvement
(19,964). Cumulative block boundaries were verified locally against the real
archive before this module was written (see
plans/260915-0955-visual-delta-fusion-pilot/reports/crosswalk-verification.md):
every block size matches Table 1 exactly, block-boundary titles change domain
at exactly the predicted positions, and for the three domains whose per-domain
5-core CSVs ship in the same archive (Arts, Games, Movies) the correspondence
`global_id == block_start + local_id` holds with zero violations and exact
ASIN-set identity.

Why ASINs: `plan.md` forbids matching items by title alone, and titles repeat
across domains. ASINs are stable identifiers, so domain attribution via ASIN
is evidence; title equality across domains is not.

Verification tiers per domain, recorded on the returned `Crosswalk`:
- `verified_by_csv`: domains whose per-domain 5-core CSVs exist in the same
  archive and passed exact zero-violation offset identity. Their local-index
  convention is proven, not assumed.
- The remaining blocks (Electronics, Home_and_Kitchen,
  Tools_and_Home_Improvement) are boundary-positional: block size and position
  match paper Table 1 to the row, but their local index files are absent from
  this archive, so `global_of` for them encodes the uniform-concatenation
  convention explicitly flagged as assumed.
"""
from __future__ import annotations

import csv
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Mapping

CATALOG_SIZE = 108753

# Domain sizes from the LLM2Rec paper, Table 1 (KDD'25), in catalog order.
# Their cumulative boundaries were verified against the real archive.
DOMAIN_SIZES: tuple[tuple[str, int], ...] = (
    ("Arts_Crafts_and_Sewing", 12454),
    ("Electronics", 20150),
    ("Home_and_Kitchen", 33478),
    ("Video_Games", 9517),
    ("Movies_and_TV", 13190),
    ("Tools_and_Home_Improvement", 19964),
)

assert sum(size for _, size in DOMAIN_SIZES) == CATALOG_SIZE

# Archive-relative per-domain CSV directories that actually ship in the
# author archive. Electronics, Home_and_Kitchen, and
# Tools_and_Home_Improvement have no per-domain files there.
ARCHIVE_DOMAIN_DIRS: Mapping[str, str] = {
    "Arts_Crafts_and_Sewing": "Arts_Crafts_and_Sewing/5-core",
    "Video_Games": "Video_Games/5-core",
    "Movies_and_TV": "Movies_and_TV/5-core",
}

DEVELOPMENT_DOMAIN = "Video_Games"
REPLICATION_DOMAIN = "Arts_Crafts_and_Sewing"


@dataclass(frozen=True)
class DomainBlock:
    """One domain's contiguous range in the global mixed catalog."""

    domain: str
    start: int
    stop: int  # exclusive

    @property
    def size(self) -> int:
        return self.stop - self.start


@dataclass(frozen=True)
class Crosswalk:
    """Validated mixed-catalog structure."""

    blocks: tuple[DomainBlock, ...]
    global_asin: Mapping[int, str]
    asin_less: frozenset[int] = frozenset()
    verified_by_csv: frozenset[str] = frozenset()

    def domain_of(self, global_id: int) -> str:
        for block in self.blocks:
            if block.start <= global_id < block.stop:
                return block.domain
        raise KeyError(f"global item {global_id} is outside the catalog")

    def global_of(self, domain: str, local_id: int) -> int:
        """Global catalog index of a domain-local 0-based item id.

        Proven exact for `verified_by_csv` domains. For the rest it encodes
        the uniform-concatenation convention (same layout as the three
        verified domains) and must be flagged `assumed` in any report or
        corpus that uses it. Position 75893 is inside the Movies block but
        has no ASIN anywhere in the archive, so it can never be returned as
        a caption donor and must be `unavailable` in every cue-bearing arm.
        """
        for block in self.blocks:
            if block.domain == domain:
                if not 0 <= local_id < block.size:
                    raise KeyError(f"{domain} local item {local_id} is out of range")
                return block.start + local_id
        raise KeyError(f"unknown domain {domain!r}")


def catalog_blocks(
    sizes: Iterable[tuple[str, int]] = DOMAIN_SIZES, catalog_size: int = CATALOG_SIZE
) -> tuple[DomainBlock, ...]:
    blocks: list[DomainBlock] = []
    cursor = 0
    for domain, size in sizes:
        if size <= 0:
            raise ValueError(f"domain {domain} has non-positive size {size}")
        blocks.append(DomainBlock(domain, cursor, cursor + size))
        cursor += size
    if cursor != catalog_size:
        raise ValueError(f"domain sizes sum to {cursor}, expected {catalog_size}")
    for previous, current in zip(blocks, blocks[1:]):
        if previous.stop != current.start:
            raise ValueError(f"gap between {previous.domain} and {current.domain}")
    return tuple(blocks)


def read_item_asin_pairs(csv_paths: Iterable[Path]) -> tuple[dict[int, str], set[int]]:
    """Read distinct `(item_id, item_asin)` pairs from 5-core CSVs (all splits).

    Returns the pairs plus the set of item ids that appear with a blank ASIN.
    A contradictory pair for one id raises instead of silently winning.
    """
    pairs: dict[int, str] = {}
    blank: set[int] = set()
    files = 0
    for csv_path in sorted(Path(path) for path in csv_paths):
        if not csv_path.is_file():
            raise FileNotFoundError(f"5-core CSV missing: {csv_path}")
        files += 1
        with csv_path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            for column in ("item_id", "item_asin"):
                if column not in (reader.fieldnames or ()):
                    raise ValueError(f"{csv_path} lacks required column {column!r}")
            for row in reader:
                raw_id = (row.get("item_id") or "").strip()
                asin = (row.get("item_asin") or "").strip()
                if not raw_id or not raw_id.isdigit():
                    continue
                item_id = int(raw_id)
                if not asin:
                    blank.add(item_id)
                    continue
                previous = pairs.setdefault(item_id, asin)
                if previous != asin:
                    raise ValueError(
                        f"{csv_path} maps item_id {item_id} to both {previous!r} and {asin!r}"
                    )
    if files == 0:
        raise ValueError("no CSV files supplied")
    return pairs, blank


def build_crosswalk(
    mixed_pairs: Mapping[int, str],
    domain_pairs: Mapping[str, Mapping[int, str]],
    blank_asin_ids: Iterable[int] = (),
    sizes: Iterable[tuple[str, int]] | None = None,
) -> Crosswalk:
    """Assemble and validate the crosswalk against the paper's Table 1.

    Every ASIN-bearing mixed id must fall in the block whose domain owns
    that ASIN. Verified domains (per-domain CSVs present) additionally
    satisfy exact `global == block_start + local` offset identity with empty
    ASIN-set difference on both sides. Positions with no ASIN anywhere are
    recorded as `asin_less` rather than attributed.
    """
    resolved = list(sizes) if sizes is not None else list(DOMAIN_SIZES)
    blocks = catalog_blocks(resolved, catalog_size=sum(size for _, size in resolved))
    by_domain = {block.domain: block for block in blocks}
    known = {domain for domain in domain_pairs if domain_pairs[domain]}
    if not known:
        raise ValueError("no per-domain item/ASIN maps supplied")

    asin_owner: dict[str, str] = {}
    for domain in known:
        if domain not in by_domain:
            raise KeyError(f"domain {domain!r} is not one of the six mix domains")
        for asin in domain_pairs[domain].values():
            owner = asin_owner.setdefault(asin, domain)
            if owner != domain:
                raise ValueError(f"ASIN {asin} appears in both {owner} and {domain}")
    catalog_size = sum(block.size for block in blocks)
    global_asin: dict[int, str] = dict(mixed_pairs)
    asin_less: set[int] = {
        global_id for global_id in range(catalog_size) if global_id not in global_asin
    }
    for global_id, asin in global_asin.items():
        owner = asin_owner.get(asin)
        if owner is None:
            continue  # Electronics/Home/Tools have no per-domain CSVs in this archive
        block = next(b for b in blocks if b.start <= global_id < b.stop)
        if owner != block.domain:
            raise ValueError(
                f"global item {global_id} ASIN {asin} belongs to {owner} "
                f"but sits in the {block.domain} block"
            )

    verified: set[str] = set()
    for domain in known:
        block = by_domain[domain]
        block_asins = {global_asin[g] for g in range(block.start, block.stop) if g in global_asin}
        csv_asins = set(domain_pairs[domain].values())
        missing_from_csv = block_asins - csv_asins
        missing_from_block = csv_asins - block_asins
        if missing_from_csv or missing_from_block:
            raise ValueError(
                f"domain {domain} ASIN sets disagree: "
                f"{len(missing_from_csv)} block ASINs absent from CSV, "
                f"{len(missing_from_block)} CSV ASINs absent from block"
            )
        csv_ids = domain_pairs[domain]
        violations = [
            local_id
            for local_id in csv_ids
            if not (0 <= local_id < block.size and global_asin.get(block.start + local_id) == csv_ids[local_id])
        ]
        if violations:
            raise ValueError(
                f"domain {domain} has {len(violations)} offset violations, first: {violations[:5]}"
            )
        verified.add(domain)

    return Crosswalk(
        blocks=blocks,
        global_asin=global_asin,
        asin_less=frozenset(asin_less),
        verified_by_csv=frozenset(verified),
    )


def bind_titles(crosswalk: Crosswalk, plain_text: str) -> list[str]:
    """Attach the positional `item_titles.txt` title list to global IDs.

    The shipped `info/item_titles.txt` is plain titles, one per line, in
    catalog order. A count mismatch raises instead of silently truncating.
    """
    lines = plain_text.split("\n")
    if lines and lines[-1] == "":
        lines.pop()
    if len(lines) != CATALOG_SIZE:
        raise ValueError(f"expected {CATALOG_SIZE} title lines, read {len(lines)}")
    return lines


def boundary_titles(titles: list[str], crosswalk: Crosswalk, width: int = 2) -> list[tuple[str, int, str]]:
    """Report the first/last `width` titles of every block for the audit."""
    report: list[tuple[str, int, str]] = []
    for block in crosswalk.blocks:
        for global_id in list(range(block.start, min(block.start + width, block.stop))):
            report.append((block.domain, global_id, titles[global_id]))
        for global_id in list(range(max(block.start, block.stop - width), block.stop)):
            report.append((block.domain, global_id, titles[global_id]))
    return report


In [ ]:
"""Captioner, paraphraser, and tokenizer backends.

Real backends (Florence-2, Qwen2.5 paraphraser, Qwen tokenizer) import
torch/transformers/PIL/huggingface_hub lazily inside their methods, never at
module import time, so this module imports cleanly on a CPU-only box with no
GPU stack installed (this workstation has none: see
plans/260915-0955-visual-delta-fusion-pilot/plan.md "Evidence and unresolved
risks"). Stub backends are what local/CPU tests exercise; they are never
substituted for the real backends on Kaggle.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Protocol, Sequence

RUNTIME_DEPENDENCIES = {"transformers": "4.44.2"}


def ensure_runtime_dependencies(expected: Mapping[str, str] | None = None) -> dict[str, str]:
    """Pin `transformers` to the version proven compatible with Florence-2's
    custom config code across this research family's completed Kaggle runs
    (`ensure_runtime_dependencies` in
    experiments/multimodal_llm_rs/kaggle/llm2rec-visual/visual_signal.py).

    Without this pin, the Kaggle image's newer default `transformers`
    raises `AttributeError: 'Florence2LanguageConfig' object has no
    attribute 'forced_bos_token_id'` inside Florence-2's own
    `configuration_florence2.py`, for both the primary and fallback caption
    model revisions alike (see
    plans/260915-0955-visual-delta-fusion-pilot/ISSUES.md #10). GPU/Kaggle
    only; call this before constructing `Florence2Captioner` or
    `QwenParaphraser`.
    """
    import importlib.metadata
    import subprocess
    import sys

    resolved_expected = dict(expected) if expected is not None else dict(RUNTIME_DEPENDENCIES)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", *(f"{name}=={version}" for name, version in resolved_expected.items())],
        check=True,
    )
    resolved = {name: importlib.metadata.version(name) for name in resolved_expected}
    if resolved != resolved_expected:
        raise RuntimeError(f"dependency resolution mismatch: expected={resolved_expected} resolved={resolved}")
    return resolved


CAPTION_MODEL_ID = "microsoft/Florence-2-large"
CAPTION_TASK_PROMPT = "<CAPTION>"
CAPTION_MAX_NEW_TOKENS = 128
CAPTION_NUM_BEAMS = 3
# Last commit before the 2024-12-08 continued-pretrain weight swap: original
# FLD-5B weights matching the model card's published benchmark table, no
# "might not be trained well" caveat. See
# plans/260915-0955-visual-delta-fusion-pilot/ISSUES.md #7 for the full
# commit-history evidence behind this pin.
CAPTION_MODEL_REVISION = "f0acedbf9b780e04fe1f9111fcf53187388f3d03"
# Current `main`: continued-pretrained 4k-context checkpoint, card-flagged as
# possibly undertrained. Only used if the primary revision fails to load.
CAPTION_MODEL_REVISION_FALLBACK = "21a599d414c4d928c9032694c424fb94458e3594"

PARAPHRASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
PARAPHRASE_MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
PARAPHRASE_MAX_NEW_TOKENS = 128
PARAPHRASE_PROMPT_TEMPLATE = (
    "Restate only the supplied product title. Preserve named entities and "
    "attributes. Add no facts or recommendations. Return one short sentence. "
    "Title: {title}"
)

TOKENIZER_MODEL_ID = "Qwen/Qwen2-0.5B"
TOKENIZER_MODEL_REVISION = "91d2aff3f957f99e4c74c962f2f408dcc88a18d8"


@dataclass(frozen=True)
class CaptionResult:
    """One raw generation result, kept separate from any later cleaning."""

    item_id: int
    raw_text: str
    status: str  # "ok" | "empty" | "decode_failure"
    generated_token_count: int | None = None


class Captioner(Protocol):
    def caption_batch(self, items: Sequence[tuple[int, Path]]) -> list[CaptionResult]: ...


class Paraphraser(Protocol):
    def paraphrase_batch(self, items: Sequence[tuple[int, str]]) -> list[CaptionResult]: ...


class Tokenizer(Protocol):
    def encode(self, text: str) -> list[int]: ...


class StubCaptioner:
    """Returns canned captions from a fixed dict. Used only by local tests."""

    def __init__(self, captions_by_item: Mapping[int, str]) -> None:
        self._captions = dict(captions_by_item)

    def caption_batch(self, items: Sequence[tuple[int, Path]]) -> list[CaptionResult]:
        results = []
        for item_id, _path in items:
            text = self._captions.get(item_id, "")
            status = "ok" if text else "empty"
            results.append(CaptionResult(item_id=item_id, raw_text=text, status=status))
        return results


class StubParaphraser:
    """Returns canned paraphrases from a fixed dict. Used only by local tests."""

    def __init__(self, paraphrases_by_item: Mapping[int, str]) -> None:
        self._paraphrases = dict(paraphrases_by_item)

    def paraphrase_batch(self, items: Sequence[tuple[int, str]]) -> list[CaptionResult]:
        results = []
        for item_id, _title in items:
            text = self._paraphrases.get(item_id, "")
            status = "ok" if text else "empty"
            results.append(CaptionResult(item_id=item_id, raw_text=text, status=status))
        return results


class StubTokenizer:
    """One token per whitespace-separated word. Used only by local tests."""

    def encode(self, text: str) -> list[int]:
        return text.split()


CAPTION_ATTN_IMPLEMENTATION = "sdpa"


def _skip_flash_attn_static_scan():
    """Context manager neutralizing transformers' static `flash_attn` import
    scan for Florence-2's remote modeling file, without touching real imports.

    Evidence (plans/260915-0955-visual-delta-fusion-pilot/ISSUES.md #12):
    Florence-2's ``modeling_florence2.py`` imports ``flash_attn`` only inside
    ``if is_flash_attn_2_available(): ...`` guards and dispatches attention
    via ``FLORENCE2_ATTENTION_CLASSES[config._attn_implementation]``; an
    ``sdpa``/``eager`` configuration never executes that branch. But
    ``transformers.dynamic_module_utils.check_imports`` performs a *static*
    source scan of every top-level import statement and raises
    ``ImportError`` for ``flash_attn`` regardless of the guard, even though
    Kaggle's T4 (sm_75) and P100 (sm_60) accelerators are both below the
    sm_80 FlashAttention-2 requires — so real installation is neither
    possible to use nor the fix. This patches only
    ``transformers.dynamic_module_utils.get_imports`` (the function the
    static scan reads its package list from) to drop ``"flash_attn"``,
    scoped to this context only, and requires the caller to pass an explicit
    non-flash ``attn_implementation`` so the omission is never silent.
    """
    from contextlib import contextmanager
    from unittest.mock import patch

    import transformers.dynamic_module_utils as dynamic_module_utils

    original_get_imports = dynamic_module_utils.get_imports

    def get_imports_without_flash_attn(filename):
        imports = original_get_imports(filename)
        return [name for name in imports if name != "flash_attn"]

    @contextmanager
    def _cm():
        with patch.object(dynamic_module_utils, "get_imports", get_imports_without_flash_attn):
            yield

    return _cm()


class Florence2Captioner:
    """Real Florence-2-large captioner. GPU/Kaggle only.

    Every heavy import (torch, transformers, PIL, huggingface_hub) is inside
    ``__init__``/``caption_batch`` so importing this module never requires
    them. ``resolved_revision`` is pinned once at construction via
    ``huggingface_hub.model_info`` and reused for every batch, never
    re-resolved from ``main`` per call (phase-01 F-requirement: "Select
    immutable HF revision once"). Loads with an explicit
    ``attn_implementation="sdpa"`` and a scoped static-scan patch (see
    ``_skip_flash_attn_static_scan``) so a genuinely unused ``flash_attn``
    import in the remote modeling file never blocks loading on
    non-Ampere Kaggle accelerators.
    """

    def __init__(self, model_id: str = CAPTION_MODEL_ID, revision: str | None = CAPTION_MODEL_REVISION) -> None:
        import torch
        from huggingface_hub import model_info
        from transformers import AutoModelForCausalLM, AutoProcessor

        self.model_id = model_id
        self.resolved_revision = revision or model_info(model_id).sha
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        with _skip_flash_attn_static_scan():
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                revision=self.resolved_revision,
                torch_dtype=self.torch_dtype,
                trust_remote_code=True,
                attn_implementation=CAPTION_ATTN_IMPLEMENTATION,
            ).to(self.device).eval()
            self.processor = AutoProcessor.from_pretrained(
                model_id, revision=self.resolved_revision, trust_remote_code=True
            )

    # Internal GPU-memory safety cap. num_beams=3 beam search internally
    # expands the effective batch by 3x during generation; a naive full
    # 64-image batch (the smoke-test call shape) measured 13.35 GiB in use
    # against a T4's 14.56 GiB before even allocating the attention matrix,
    # producing a real `CUDA out of memory` error (issue #25 verification
    # run). Every caller — this fixed constant, not the caller's list size —
    # controls the actual forward-pass batch, so existing 64/300-item
    # callers (`real_generator_smoke`, `quality_review`) stay correct.
    MAX_GPU_BATCH = 4

    def caption_batch(self, items: Sequence[tuple[int, Path]]) -> list[CaptionResult]:
        import torch
        from PIL import Image

        results: list[CaptionResult] = []
        valid_ids: list[int] = []
        valid_images: list["Image.Image"] = []
        for item_id, image_path in items:
            try:
                image = Image.open(image_path).convert("RGB")
            except Exception:
                results.append(CaptionResult(item_id=item_id, raw_text="", status="decode_failure"))
                continue
            valid_ids.append(item_id)
            valid_images.append(image)
        for start in range(0, len(valid_images), self.MAX_GPU_BATCH):
            chunk_ids = valid_ids[start:start + self.MAX_GPU_BATCH]
            chunk_images = valid_images[start:start + self.MAX_GPU_BATCH]
            # Every image shares the identical `<CAPTION>` prompt, so the
            # text side never needs padding; only pixel_values stack.
            inputs = self.processor(
                text=[CAPTION_TASK_PROMPT] * len(chunk_images), images=chunk_images, return_tensors="pt", padding=True
            ).to(self.device, self.torch_dtype)
            with torch.no_grad():
                generated_ids = self.model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    max_new_tokens=CAPTION_MAX_NEW_TOKENS,
                    num_beams=CAPTION_NUM_BEAMS,
                    do_sample=False,
                )
            prompt_length = inputs["input_ids"].shape[-1]
            pad_id = self.model.generation_config.pad_token_id
            for item_id, image, row in zip(chunk_ids, chunk_images, generated_ids):
                # `generate()` right-pads every row in the batch to the
                # longest generated sequence with `pad_token_id`. Trim only
                # that trailing padding (never a legitimate generated
                # token) so each row is decoded with the exact same
                # `skip_special_tokens=False` semantics the proven
                # single-item path used (issue #13's verified
                # `status: PASS` run) — batching must not change what
                # `post_process_generation` receives for any one image.
                end = row.shape[-1]
                if pad_id is not None:
                    while end > prompt_length and int(row[end - 1]) == pad_id:
                        end -= 1
                trimmed_row = row[:end]
                generated_text = self.processor.batch_decode(trimmed_row.unsqueeze(0), skip_special_tokens=False)[0]
                parsed = self.processor.post_process_generation(
                    generated_text, task=CAPTION_TASK_PROMPT, image_size=(image.width, image.height)
                )
                raw_text = str(parsed.get(CAPTION_TASK_PROMPT, "")).strip()
                status = "ok" if raw_text else "empty"
                results.append(CaptionResult(
                    item_id=item_id, raw_text=raw_text, status=status,
                    generated_token_count=int(end - prompt_length),
                ))
            del inputs, generated_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return results


class QwenParaphraser:
    """Real title-only paraphrase control. GPU/Kaggle only.

    Sees only the title text — never the image, caption, or any interaction
    data — so it cannot leak visual information into the paraphrase control
    (plan.md: "a separate frozen text model used ONLY to build a control").
    """

    def __init__(self, model_id: str = PARAPHRASE_MODEL_ID, revision: str | None = PARAPHRASE_MODEL_REVISION) -> None:
        import torch
        from huggingface_hub import model_info
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.model_id = model_id
        self.resolved_revision = revision or model_info(model_id).sha
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, revision=self.resolved_revision)
        # Left padding is required for correct batched causal-LM generation:
        # it keeps every sequence's real content right-aligned, so the
        # generated continuation starts at the same column index for every
        # row regardless of prompt length.
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, revision=self.resolved_revision, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        ).to(self.device).eval()

    # Same GPU-memory safety rationale as `Florence2Captioner.MAX_GPU_BATCH`:
    # this fixed constant, not the caller's list size, controls the actual
    # forward-pass batch, so existing 64/300-item callers stay correct.
    MAX_GPU_BATCH = 4

    def paraphrase_batch(self, items: Sequence[tuple[int, str]]) -> list[CaptionResult]:
        import torch

        results: list[CaptionResult] = []
        for start in range(0, len(items), self.MAX_GPU_BATCH):
            chunk = items[start:start + self.MAX_GPU_BATCH]
            prompts = [
                self.tokenizer.apply_chat_template(
                    [{"role": "user", "content": PARAPHRASE_PROMPT_TEMPLATE.format(title=title)}],
                    add_generation_prompt=True, tokenize=False,
                )
                for _item_id, title in chunk
            ]
            encoded = self.tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.device)
            with torch.no_grad():
                generated_ids = self.model.generate(
                    **encoded, max_new_tokens=PARAPHRASE_MAX_NEW_TOKENS, do_sample=False
                )
            # Left padding means every prompt (real content plus left pad)
            # occupies exactly `encoded["input_ids"].shape[-1]` columns, so
            # the newly generated continuation is that same uniform suffix
            # for every row in the batch — the standard batched-generation
            # slicing pattern, unaffected by each prompt's real (unpadded)
            # length.
            prompt_length = encoded["input_ids"].shape[-1]
            new_tokens_batch = generated_ids[:, prompt_length:]
            pad_id = self.tokenizer.pad_token_id
            for (item_id, _title), new_tokens in zip(chunk, new_tokens_batch):
                text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
                status = "ok" if text else "empty"
                token_count = int((new_tokens != pad_id).sum()) if pad_id is not None else int(new_tokens.shape[-1])
                results.append(CaptionResult(
                    item_id=item_id, raw_text=text, status=status,
                    generated_token_count=token_count,
                ))
            del encoded, generated_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return results


class QwenTokenizer:
    """Real Qwen2 tokenizer wrapper for token-cap budgeting. GPU/Kaggle only."""

    def __init__(self, model_id: str = TOKENIZER_MODEL_ID, revision: str | None = TOKENIZER_MODEL_REVISION) -> None:
        from huggingface_hub import model_info
        from transformers import AutoTokenizer

        self.model_id = model_id
        self.resolved_revision = revision or model_info(model_id).sha
        self._tokenizer = AutoTokenizer.from_pretrained(model_id, revision=self.resolved_revision)

    def encode(self, text: str) -> list[int]:
        return self._tokenizer.encode(text, add_special_tokens=False)


In [ ]:
"""Real generator smoke: 64 images across the six AmazonMix-6 domains.

Phase 2 Implementation Step 4. Real Florence-2 caption generation and real
Qwen2.5 title-only paraphrasing on real, decoded images and real catalog
titles — no stub backend anywhere in this module. GPU/Kaggle only; every
heavy import stays inside `caption.py`'s lazy backends.
"""
from __future__ import annotations

import ast
import csv
import gzip
import hashlib
import io
import json
import shutil
import time
import urllib.request
from pathlib import Path
from typing import Callable

USER_AGENT = "llm2rec-caption-augmentation-smoke/1.0"
SAMPLE_TOTAL = 64
IMAGE_MAX_BYTES = 8 * 1024 * 1024
METADATA_URL_TEMPLATE = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/"
    "meta_categories/meta_{category}.jsonl.gz"
)


def _evenly_spaced_positions(start: int, size: int, count: int) -> list[int]:
    if count <= 0:
        return []
    if count == 1:
        return [start + size // 2]
    return [start + round(index * (size - 1) / (count - 1)) for index in range(count)]


def select_smoke_sample(total: int = SAMPLE_TOTAL) -> list[tuple[str, int]]:
    """Deterministic sample of `total` global IDs spread across all six domains.

    Evenly spaced positions within each domain's block, never "first IDs"
    (phase-02-kaggle-package.md Step 4). Earlier domains in catalog order
    absorb the remainder when `total` does not divide evenly by six.
    """
    blocks = catalog_blocks()
    domain_count = len(blocks)
    base, remainder = divmod(total, domain_count)
    counts = [base + (1 if index < remainder else 0) for index in range(domain_count)]
    sample: list[tuple[str, int]] = []
    for block, count in zip(blocks, counts):
        for global_id in _evenly_spaced_positions(block.start, block.size, count):
            sample.append((block.domain, global_id))
    return sample


def select_quality_review_sample(per_domain: int = 50) -> list[tuple[str, int]]:
    """Deterministic sample of `per_domain` global IDs per domain, evenly
    spaced (never "first IDs"), selected before generation
    (phase-01-local-module.md Step 6: "Deterministic sample of 50 items per
    pretraining domain, selected before generation").
    """
    sample: list[tuple[str, int]] = []
    for block in catalog_blocks():
        for global_id in _evenly_spaced_positions(block.start, block.size, per_domain):
            sample.append((block.domain, global_id))
    return sample


def locate_mixed_root(search_root: Path = Path("/kaggle/input")) -> Path:
    candidates = sorted(search_root.glob("**/AmazonMix-6/5-core"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"expected exactly one mounted AmazonMix-6/5-core directory, found {candidates}"
        )
    return candidates[0]


def load_asin_titles(mixed_root: Path, wanted_ids: set[int]) -> dict[int, tuple[str, str]]:
    """Return `{global_id: (asin, title)}` for exactly the wanted global IDs.

    A wanted ID absent from every split's ASIN column is omitted here (the
    catalog's own 20 ASIN-less positions, see
    plans/260915-0955-visual-delta-fusion-pilot/reports/crosswalk-verification.md);
    callers must record it as `no_asin_in_catalog`, never drop it silently.
    """
    pairs, _blank = read_item_asin_pairs(list(mixed_root.glob("*/*.csv")))
    titles_path = mixed_root / "info" / "item_titles.txt"
    lines = titles_path.read_text(encoding="utf-8").split("\n")
    if lines and lines[-1] == "":
        lines.pop()
    result: dict[int, tuple[str, str]] = {}
    for global_id in wanted_ids:
        asin = pairs.get(global_id)
        if asin is None:
            continue
        result[global_id] = (asin, lines[global_id])
    return result


def choose_image_url(record: dict) -> str | None:
    images = record.get("images") or []
    for image in images:
        if not isinstance(image, dict):
            continue
        for key in ("hi_res", "large", "thumb"):
            value = image.get(key)
            if isinstance(value, str) and value.startswith("http"):
                return value
    return None


def fetch_domain_image_urls(category: str, wanted_asins: set[str], cache_dir: Path) -> dict[str, str]:
    """Stream `meta_<category>.jsonl.gz`, returning `{asin: image_url}` found.

    Stops early once every wanted ASIN in this category is found; the
    metadata file order is not guaranteed, so a worst-case target near the
    file's end still requires a full stream.
    """
    if not wanted_asins:
        return {}
    cache_dir.mkdir(parents=True, exist_ok=True)
    archive_path = cache_dir / f"meta_{category}.jsonl.gz"
    if not archive_path.is_file() or archive_path.stat().st_size == 0:
        url = METADATA_URL_TEMPLATE.format(category=category)
        request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
        temp_path = archive_path.with_suffix(".jsonl.gz.partial")
        with urllib.request.urlopen(request, timeout=180) as response, temp_path.open("wb") as output:
            shutil.copyfileobj(response, output)
        temp_path.replace(archive_path)
    remaining = set(wanted_asins)
    found: dict[str, str] = {}
    with gzip.open(archive_path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if not remaining:
                break
            record = json.loads(line)
            parent_asin = record.get("parent_asin")
            if parent_asin in remaining:
                url = choose_image_url(record)
                if url:
                    found[parent_asin] = url
                remaining.discard(parent_asin)
    return found


def download_image(url: str, dest: Path) -> dict[str, object]:
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    last_error: str | None = None
    for attempt in range(3):
        try:
            with urllib.request.urlopen(request, timeout=30) as response:
                payload = response.read(IMAGE_MAX_BYTES + 1)
            if len(payload) > IMAGE_MAX_BYTES:
                return {"status": "download_failed", "error": "exceeds_size_limit"}
            from PIL import Image

            with Image.open(io.BytesIO(payload)) as image:
                image.convert("RGB")
            dest.write_bytes(payload)
            return {
                "status": "decoded",
                "path": str(dest),
                "bytes": len(payload),
                "sha256": hashlib.sha256(payload).hexdigest(),
            }
        except Exception as exc:  # noqa: BLE001 - preserve per-image failure evidence
            last_error = f"{type(exc).__name__}: {exc}"
    return {"status": "download_failed", "error": last_error}


def run_generation_pass(
    sample: list[tuple[str, int]],
    work_dir: Path,
    mixed_root: Path,
    stage: str,
    output_filename: str,
    temp_dir: Path = Path("/kaggle/temp"),
) -> dict[str, object]:
    """Shared real-generation core for both the 64-item smoke and the
    Phase 1 Step 6 quality-review sample (50 items/domain). Identical logic,
    different sample size and output name; nothing about caption/paraphrase
    generation differs between the two.
    """
    t_start = time.perf_counter()
    wanted_ids = {global_id for _domain, global_id in sample}
    asin_titles = load_asin_titles(mixed_root, wanted_ids)

    records: list[dict[str, object]] = []
    by_domain: dict[str, list[tuple[int, str, str]]] = {}
    for domain, global_id in sample:
        entry = asin_titles.get(global_id)
        if entry is None:
            records.append({
                "domain": domain, "global_id": global_id, "asin": None, "title": None,
                "image_url": None, "image_status": "no_asin_in_catalog",
            })
            continue
        asin, title = entry
        by_domain.setdefault(domain, []).append((global_id, asin, title))

    # Metadata and images are scratch data: written under /kaggle/temp, never
    # under work_dir (/kaggle/working), so they are never bundled into the
    # kernel's persisted output. Each domain's multi-GB meta_*.jsonl.gz is
    # deleted immediately after its URLs are extracted, not accumulated
    # across all six domains, to bound peak disk usage.
    cache_dir = temp_dir / f"{stage}_metadata_cache"
    image_dir = temp_dir / f"{stage}_images"
    image_dir.mkdir(parents=True, exist_ok=True)

    t_metadata_start = time.perf_counter()
    for domain, entries in by_domain.items():
        wanted_asins = {asin for _gid, asin, _title in entries}
        url_map = fetch_domain_image_urls(domain, wanted_asins, cache_dir)
        archive_path = cache_dir / f"meta_{domain}.jsonl.gz"
        archive_path.unlink(missing_ok=True)
        for global_id, asin, title in entries:
            url = url_map.get(asin)
            records.append({
                "domain": domain, "global_id": global_id, "asin": asin, "title": title,
                "image_url": url,
                "image_status": "missing_metadata" if url is None else "pending",
            })
    metadata_seconds = time.perf_counter() - t_metadata_start

    t_download_start = time.perf_counter()
    for record in records:
        if record.get("image_status") != "pending":
            continue
        dest = image_dir / f"{record['global_id']}.bin"
        outcome = download_image(str(record["image_url"]), dest)
        record["image_status"] = outcome["status"]
        if outcome["status"] == "decoded":
            record["image_path"] = outcome["path"]
            record["image_bytes"] = outcome["bytes"]
            record["image_sha256"] = outcome["sha256"]
        else:
            record["image_error"] = outcome.get("error")
    download_seconds = time.perf_counter() - t_download_start

    caption_items = [
        (record["global_id"], Path(str(record["image_path"])))
        for record in records if record.get("image_status") == "decoded"
    ]

    t_caption_load_start = time.perf_counter()
    revision_used = CAPTION_MODEL_REVISION
    revision_fallback_triggered = False
    revision_error: str | None = None
    try:
        captioner = Florence2Captioner(revision=CAPTION_MODEL_REVISION)
    except Exception as exc:  # noqa: BLE001 - explicit, reported fallback only
        revision_error = f"{type(exc).__name__}: {exc}"
        revision_used = CAPTION_MODEL_REVISION_FALLBACK
        revision_fallback_triggered = True
        captioner = Florence2Captioner(revision=CAPTION_MODEL_REVISION_FALLBACK)
    caption_model_load_seconds = time.perf_counter() - t_caption_load_start

    t_caption_start = time.perf_counter()
    caption_results = captioner.caption_batch(caption_items)
    caption_seconds = time.perf_counter() - t_caption_start

    caption_by_id = {result.item_id: result for result in caption_results}
    for record in records:
        result = caption_by_id.get(record["global_id"])
        if result is None:
            continue
        record["caption_status"] = result.status
        record["caption_raw"] = result.raw_text
        record["caption_generated_tokens"] = result.generated_token_count
        record["caption_hit_token_cap"] = result.generated_token_count == CAPTION_MAX_NEW_TOKENS

    paraphrase_items = [
        (record["global_id"], str(record["title"]))
        for record in records if record.get("title")
    ]
    t_paraphrase_load_start = time.perf_counter()
    paraphraser = QwenParaphraser()
    paraphrase_model_load_seconds = time.perf_counter() - t_paraphrase_load_start
    t_paraphrase_start = time.perf_counter()
    paraphrase_results = paraphraser.paraphrase_batch(paraphrase_items)
    paraphrase_seconds = time.perf_counter() - t_paraphrase_start
    paraphrase_by_id = {result.item_id: result for result in paraphrase_results}
    for record in records:
        result = paraphrase_by_id.get(record["global_id"])
        if result is None:
            continue
        record["paraphrase_status"] = result.status
        record["paraphrase_text"] = result.raw_text
        record["paraphrase_generated_tokens"] = result.generated_token_count
        record["paraphrase_hit_token_cap"] = result.generated_token_count == PARAPHRASE_MAX_NEW_TOKENS

    decoded_count = sum(1 for record in records if record.get("image_status") == "decoded")
    captioned_ok = sum(1 for record in records if record.get("caption_status") == "ok")
    paraphrased_ok = sum(1 for record in records if record.get("paraphrase_status") == "ok")
    caption_cap_hits = sum(1 for record in records if record.get("caption_hit_token_cap"))
    paraphrase_cap_hits = sum(1 for record in records if record.get("paraphrase_hit_token_cap"))

    result: dict[str, object] = {
        "status": "PASS" if decoded_count > 0 and captioned_ok > 0 and paraphrased_ok > 0 else "FAIL",
        "stage": stage,
        "sample_size": len(sample),
        "domains_sampled": sorted(by_domain),
        "image_coverage": decoded_count / len(sample),
        "caption_ok_rate": captioned_ok / max(decoded_count, 1),
        "caption_token_cap_hit_rate": caption_cap_hits / max(decoded_count, 1),
        "paraphrase_ok_rate": paraphrased_ok / max(len(paraphrase_items), 1),
        "paraphrase_token_cap_hit_rate": paraphrase_cap_hits / max(len(paraphrase_items), 1),
        "caption_model_revision_used": revision_used,
        "caption_model_revision_fallback_triggered": revision_fallback_triggered,
        "caption_model_revision_primary_error": revision_error,
        "timing_seconds": {
            "metadata_stream": metadata_seconds,
            "image_download": download_seconds,
            "caption_model_load": caption_model_load_seconds,
            "caption_generation_total": caption_seconds,
            "caption_generation_per_image": caption_seconds / max(decoded_count, 1),
            "paraphrase_model_load": paraphrase_model_load_seconds,
            "paraphrase_generation_total": paraphrase_seconds,
            "paraphrase_generation_per_item": paraphrase_seconds / max(len(paraphrase_items), 1),
            "total": time.perf_counter() - t_start,
        },
        "records": records,
    }
    (work_dir / output_filename).write_text(json.dumps(result, indent=2, sort_keys=True), encoding="utf-8")
    summary = {key: value for key, value in result.items() if key != "records"}
    print(json.dumps(summary, indent=2, sort_keys=True))
    return result


def run_real_generator_smoke(
    work_dir: Path, mixed_root: Path, temp_dir: Path = Path("/kaggle/temp")
) -> dict[str, object]:
    return run_generation_pass(
        select_smoke_sample(), work_dir, mixed_root, "real_generator_smoke",
        "smoke_results.json", temp_dir,
    )


def run_quality_review(
    work_dir: Path, mixed_root: Path, temp_dir: Path = Path("/kaggle/temp"), per_domain: int = 50
) -> dict[str, object]:
    """Phase 1 Step 6: 50-items-per-domain deterministic sample, real
    captions and paraphrases, written for human review. This function
    generates the review material; the unsupported-attribute/redundant/
    adds-visible-attribute/unclear labels remain human judgments per
    phase-01-local-module.md ("These are human-reviewed judgments, not
    automatic semantic truth").
    """
    return run_generation_pass(
        select_quality_review_sample(per_domain), work_dir, mixed_root, "quality_review",
        "quality_review_results.json", temp_dir,
    )


def run_paraphrase_probe(
    work_dir: Path,
    mixed_root: Path,
    model_id: str = PARAPHRASE_MODEL_ID,
    revision: str | None = PARAPHRASE_MODEL_REVISION,
    per_domain: int = 50,
    output_filename: str = "paraphrase_probe_results.json",
) -> dict[str, object]:
    """Paraphraser-only comparison probe: the same 300-item deterministic
    quality-review sample and titles as `run_quality_review`, but skips
    metadata streaming, image download, and Florence-2 captioning entirely
    (paraphrasing is title-only and does not depend on any of them). Used to
    A/B a candidate paraphraser model against the pinned default without
    paying the ~470s metadata + ~80s image + ~165s caption cost every time.

    See plans/260915-0955-visual-delta-fusion-pilot/ISSUES.md #15 for why
    this comparison is needed: the pinned Qwen2.5-0.5B-Instruct paraphraser
    hallucinates specific unsupported facts on ~18% of a real 300-item
    sample (worst domain ~56%), far outside the protocol's proposed ≤5%
    gate.
    """
    t_start = time.perf_counter()
    sample = select_quality_review_sample(per_domain)
    wanted_ids = {global_id for _domain, global_id in sample}
    asin_titles = load_asin_titles(mixed_root, wanted_ids)

    records: list[dict[str, object]] = []
    for domain, global_id in sample:
        entry = asin_titles.get(global_id)
        if entry is None:
            records.append({
                "domain": domain, "global_id": global_id, "title": None,
                "paraphrase_status": "no_asin_in_catalog",
            })
            continue
        _asin, title = entry
        records.append({"domain": domain, "global_id": global_id, "title": title})

    paraphrase_items = [
        (record["global_id"], str(record["title"]))
        for record in records if record.get("title")
    ]
    t_load_start = time.perf_counter()
    paraphraser = QwenParaphraser(model_id=model_id, revision=revision)
    model_load_seconds = time.perf_counter() - t_load_start

    t_gen_start = time.perf_counter()
    paraphrase_results = paraphraser.paraphrase_batch(paraphrase_items)
    generation_seconds = time.perf_counter() - t_gen_start

    paraphrase_by_id = {result.item_id: result for result in paraphrase_results}
    for record in records:
        result = paraphrase_by_id.get(record["global_id"])
        if result is None:
            continue
        record["paraphrase_status"] = result.status
        record["paraphrase_text"] = result.raw_text
        record["paraphrase_generated_tokens"] = result.generated_token_count
        record["paraphrase_hit_token_cap"] = result.generated_token_count == PARAPHRASE_MAX_NEW_TOKENS

    paraphrased_ok = sum(1 for record in records if record.get("paraphrase_status") == "ok")
    cap_hits = sum(1 for record in records if record.get("paraphrase_hit_token_cap"))

    result: dict[str, object] = {
        "status": "PASS" if paraphrased_ok > 0 else "FAIL",
        "stage": "paraphrase_probe",
        "model_id": model_id,
        "resolved_revision": paraphraser.resolved_revision,
        "sample_size": len(sample),
        "paraphrase_ok_rate": paraphrased_ok / max(len(paraphrase_items), 1),
        "paraphrase_token_cap_hit_rate": cap_hits / max(len(paraphrase_items), 1),
        "timing_seconds": {
            "model_load": model_load_seconds,
            "generation_total": generation_seconds,
            "generation_per_item": generation_seconds / max(len(paraphrase_items), 1),
            "total": time.perf_counter() - t_start,
        },
        "records": records,
    }
    (work_dir / output_filename).write_text(json.dumps(result, indent=2, sort_keys=True), encoding="utf-8")
    summary = {key: value for key, value in result.items() if key != "records"}
    print(json.dumps(summary, indent=2, sort_keys=True))
    return result


def resolve_full_catalog_metadata(
    mixed_root: Path,
    temp_dir: Path = Path("/kaggle/temp"),
    progress: Callable[[str], None] = print,
) -> tuple[list[str], list[dict[str, object]]]:
    """Resolve every catalog item's ASIN/title/image URL, one domain at a
    time, with real per-domain progress. Returns the full `item_titles.txt`
    list and a per-item metadata record (`global_id`, `domain`, `title`,
    `asin`, `image_url`, `image_status`) in catalog order, ready for
    `process_catalog_records_incrementally`.

    Split out from the old monolithic `run_full_corpus_generation` (see
    plans/260915-0955-visual-delta-fusion-pilot/ISSUES.md #23) so metadata
    resolution prints visible progress instead of running silently for
    minutes with no indication of where it is.
    """
    titles_path = mixed_root / "info" / "item_titles.txt"
    titles = titles_path.read_text(encoding="utf-8").split("\n")
    if titles and titles[-1] == "":
        titles.pop()
    if len(titles) != CATALOG_SIZE:
        raise RuntimeError(f"catalog title count {len(titles)} != {CATALOG_SIZE}")
    all_ids = set(range(CATALOG_SIZE))
    asin_titles = load_asin_titles(mixed_root, all_ids)
    metadata_dir = temp_dir / "full_corpus_metadata"
    records: list[dict[str, object] | None] = [None] * CATALOG_SIZE
    for block in catalog_blocks():
        t0 = time.perf_counter()
        domain_ids = range(block.start, block.stop)
        asins = {asin_titles[item_id][0] for item_id in domain_ids if item_id in asin_titles}
        url_by_asin = fetch_domain_image_urls(block.domain, asins, metadata_dir)
        (metadata_dir / f"meta_{block.domain}.jsonl.gz").unlink(missing_ok=True)
        resolved = 0
        for global_id in domain_ids:
            entry = asin_titles.get(global_id)
            record: dict[str, object] = {"global_id": global_id, "domain": block.domain, "title": titles[global_id]}
            if entry is None:
                record["image_status"] = "no_asin_in_catalog"
            else:
                asin, _title = entry
                record["asin"] = asin
                image_url = url_by_asin.get(asin)
                if image_url is None:
                    record["image_status"] = "missing_metadata"
                else:
                    record["image_url"] = image_url
                    record["image_status"] = "pending"
                    resolved += 1
            records[global_id] = record
        progress(
            f"metadata: {block.domain:26s} {resolved}/{block.size} image URLs resolved "
            f"in {time.perf_counter() - t0:.1f}s"
        )
    return titles, records  # type: ignore[return-value]


def process_catalog_records_incrementally(
    metadata_records: list[dict[str, object]],
    captioner: Captioner,
    paraphraser: Paraphraser,
    out_dir: Path,
    identity: dict[str, object],
    image_dir: Path,
    resolve_image: Callable[[dict[str, object], Path], dict[str, object]],
    shard_size: int = 512,
    progress: Callable[[str], None] = print,
    deadline: float | None = None,
    batch_size: int = 8,
) -> list[dict[str, object]]:
    """Caption+paraphrase every catalog record in `batch_size`-sized chunks,
    flushing a shard checkpoint at least every `shard_size` records so a
    cancelled run loses only a bounded amount of work, and resuming
    automatically from any existing, validated checkpoint whose identity
    matches.

    This is the fix for plans/260915-0955-visual-delta-fusion-pilot/ISSUES.md
    #23 (checkpointing) and #25 (throughput): the original implementation
    both wrote its checkpoint only once at the end AND called
    `caption_batch`/`paraphrase_batch` with a single-item list every time,
    even though both accept a list — the measured real throughput
    (0.60 items/s) confirmed the single-item call pattern was the
    bottleneck, not the model itself. Chunking `batch_size` items per
    `caption_batch`/`paraphrase_batch` call lets `Florence2Captioner` and
    `QwenParaphraser` actually batch the GPU forward pass instead of
    looping one example at a time. `resolve_image` is injected so this
    function is fully testable offline with a fake, network-free resolver
    and `StubCaptioner`/`StubParaphraser`.

    `deadline` is an optional `time.perf_counter()`-comparable timestamp
    (see issue #23/#24: Kaggle enforces a hard ~21,600s script execution
    cap). It is checked once per chunk (not per item, since a chunk is now
    one atomic unit of GPU work) and, on expiry, checkpoints whatever is
    done so far and returns early rather than letting the platform kill the
    process mid-chunk with an unflushed partial shard.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "shard-manifest.json"
    records: list[dict[str, object]] = []
    total = len(metadata_records)
    if manifest_path.is_file():
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        if existing.get("identity") != dict(identity):
            raise RuntimeError(
                f"existing checkpoint at {out_dir} has a different run identity; "
                "use a fresh output directory for a new identity instead of overwriting evidence"
            )
        records = load_shard_records(out_dir, identity)
        progress(f"resuming from checkpoint: {len(records)}/{total} records already complete")
    start_index = len(records)
    if start_index >= total:
        progress(f"checkpoint already covers the full catalog: {start_index}/{total}")
        return records
    t_start = time.perf_counter()
    last_checkpoint = start_index
    for chunk_start in range(start_index, total, batch_size):
        if deadline is not None and time.perf_counter() >= deadline:
            progress(
                f"time budget reached at {chunk_start}/{total} records; stopping "
                "and checkpointing for a later resumed run"
            )
            break
        chunk_end = min(chunk_start + batch_size, total)
        chunk_records: list[dict[str, object]] = []
        image_paths: dict[int, Path] = {}
        for index in range(chunk_start, chunk_end):
            record = dict(metadata_records[index])
            if record.get("image_status") == "pending":
                outcome = resolve_image(record, image_dir)
                record["image_status"] = outcome["status"]
                if outcome["status"] == "decoded":
                    record["image_sha256"] = outcome["sha256"]
                    image_paths[record["global_id"]] = Path(str(outcome["path"]))
                else:
                    record["image_error"] = outcome.get("error")
            chunk_records.append(record)
        eligible = [
            record for record in chunk_records
            if record.get("image_status") == "decoded" and record.get("title") and record["global_id"] in image_paths
        ]
        if eligible:
            caption_items = [(record["global_id"], image_paths[record["global_id"]]) for record in eligible]
            caption_results = {
                result.item_id: result
                for result in captioner.caption_batch(caption_items)
            }
            paraphrase_results = {
                result.item_id: result
                for result in paraphraser.paraphrase_batch(
                    [(record["global_id"], str(record["title"])) for record in eligible]
                )
            }
            for record in eligible:
                caption_result = caption_results.get(record["global_id"])
                paraphrase_result = paraphrase_results.get(record["global_id"])
                if caption_result is not None:
                    record["caption_status"] = caption_result.status
                    record["caption_raw"] = caption_result.raw_text
                if paraphrase_result is not None:
                    record["paraphrase_status"] = paraphrase_result.status
                    record["paraphrase_text"] = paraphrase_result.raw_text
        records.extend(chunk_records)
        completed = len(records)
        if completed - last_checkpoint >= shard_size or completed == total:
            write_sharded_records(records, out_dir, identity, shard_size)
            last_checkpoint = completed
            elapsed = time.perf_counter() - t_start
            rate = (completed - start_index) / max(elapsed, 1e-9)
            eta_minutes = (total - completed) / max(rate, 1e-9) / 60
            progress(
                f"checkpoint: {completed}/{total} ({completed / total:.1%}) "
                f"elapsed={elapsed:.1f}s rate={rate:.2f}/s eta={eta_minutes:.1f}min"
            )
    if len(records) > last_checkpoint:
        # Deadline hit between checkpoints: flush the partial tail so
        # nothing since the last checkpoint is lost.
        write_sharded_records(records, out_dir, identity, shard_size)
        progress(f"final checkpoint before stopping: {len(records)}/{total} records saved")
    return records


def run_full_corpus_generation(
    work_dir: Path,
    mixed_root: Path,
    temp_dir: Path = Path("/kaggle/temp"),
    shard_size: int = 512,
    progress: Callable[[str], None] = print,
    time_budget_seconds: float | None = None,
) -> dict[str, object]:
    """Generate the complete six-domain catalog and persisted arm shards.

    Orchestrates `resolve_full_catalog_metadata` and
    `process_catalog_records_incrementally` (checkpointed, resumable), then
    builds the five arms once every record is captioned/paraphrased.

    `time_budget_seconds`, when set, caps the generation loop's own
    wall-clock time (see `process_catalog_records_incrementally`'s
    `deadline`). If the budget expires before the whole catalog is
    generated, this function returns an explicit `status: "partial"`
    summary and skips arm assembly — arms need the complete captioned set,
    and are correctly deferred to a later call that resumes from this run's
    checkpoint and finishes the catalog.
    """
    t_start = time.perf_counter()
    _titles, metadata_records = resolve_full_catalog_metadata(mixed_root, temp_dir, progress=progress)
    captioner = Florence2Captioner(revision=CAPTION_MODEL_REVISION)
    paraphraser = QwenParaphraser(model_id=PARAPHRASE_MODEL_ID, revision=PARAPHRASE_MODEL_REVISION)
    image_dir = temp_dir / "full_corpus_images"
    image_dir.mkdir(parents=True, exist_ok=True)

    def resolve_image(record: dict[str, object], image_dir: Path) -> dict[str, object]:
        dest = image_dir / f"{record['global_id']}.bin"
        return download_image(str(record["image_url"]), dest)

    identity = {
        "stage": "full_corpus_generation",
        "catalog_size": CATALOG_SIZE,
        "paraphrase_model": PARAPHRASE_MODEL_ID,
        "paraphrase_revision": PARAPHRASE_MODEL_REVISION,
        "caption_revision": CAPTION_MODEL_REVISION,
        "shuffle_seed": 7001,
        "shard_size": shard_size,
    }
    out_dir = work_dir / "full_corpus"
    deadline = t_start + time_budget_seconds if time_budget_seconds is not None else None
    records = process_catalog_records_incrementally(
        metadata_records, captioner, paraphraser, out_dir, identity, image_dir, resolve_image,
        shard_size=shard_size, progress=progress, deadline=deadline,
    )

    if len(records) < len(metadata_records):
        status = {
            "status": "partial",
            "identity": identity,
            "record_count": len(records),
            "catalog_size": len(metadata_records),
            "note": "time budget expired before the full catalog was generated; "
                    "re-run the same identity to resume and finish generation before arm assembly",
        }
        progress(json.dumps(status, indent=2, sort_keys=True))
        return status


    train_files = sorted((mixed_root / "train").glob("*.csv"))
    sequences: list[list[int]] = []
    for path in train_files:
        with path.open(encoding="utf-8", newline="") as handle:
            for row in csv.DictReader(handle):
                try:
                    sequences.append([int(value) for value in ast.literal_eval(row["history_item_id"])])
                except (KeyError, SyntaxError, ValueError):
                    continue
    frequencies = compute_training_frequencies(sequences)
    real_cues = {record["global_id"]: record["caption_raw"] for record in records if record.get("caption_status") == "ok"}
    paraphrase_cues = {
        record["global_id"]: record["paraphrase_text"] for record in records if record.get("paraphrase_status") == "ok"
    }
    donor_map = frequency_bin_derangement(sorted(real_cues), frequencies, seed=7001)
    rows = [
        CatalogRow(
            domain=str(record["domain"]), source_item_id=str(record["global_id"]),
            downstream_item_id=record["global_id"], parent_asin=record.get("asin"),
            source_title=str(record["title"]), order_index=record["global_id"],
            image_status=str(record.get("image_status", "unknown")),
            caption_status=str(record.get("caption_status", "unknown")),
            raw_caption=record.get("caption_raw"),
        )
        for record in records
    ]
    arms = build_arms(rows, real_cues, paraphrase_cues, donor_map)
    for record, arm in zip(records, arms):
        record["arm_texts"] = arm.texts
        record["shuffle_donor_id"] = arm.shuffle_donor_id

    manifest = write_sharded_records(records, out_dir, identity, shard_size)
    manifest["available_caption_count"] = len(real_cues)
    manifest["image_decoded_count"] = sum(record.get("image_status") == "decoded" for record in records)
    manifest["paraphrase_ok_count"] = sum(record.get("paraphrase_status") == "ok" for record in records)
    (work_dir / "full_corpus_summary.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    progress(json.dumps({key: value for key, value in manifest.items() if key != "shards"}, indent=2, sort_keys=True))
    return manifest


In [ ]:
import time

KERNEL_START = time.perf_counter()
resolved = ensure_runtime_dependencies()
print("pinned runtime dependencies:", resolved)


In [ ]:
mixed_root = locate_mixed_root()
print("mixed_root =", mixed_root)
_titles, metadata_records = resolve_full_catalog_metadata(mixed_root)
print(f"metadata resolved for {len(metadata_records)} catalog items")


In [ ]:
import hashlib
import json
import shutil
import time
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
TEMP_DIR = Path("/kaggle/temp")
OUT_DIR = WORK_DIR / "full_corpus"
IMAGE_DIR = TEMP_DIR / "full_corpus_images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# /kaggle/working starts empty on every single push; a checkpoint written
# by an earlier version of THIS SAME kernel only survives if that version's
# output is mounted as a `kernel_sources` input (self-reference) and copied
# back into /kaggle/working here. Located by filename, not by an assumed
# mount path, per rule://kaggle-mcp-experiments. The first push has no
# kernel_sources yet, so this is a no-op and generation starts from item 0.
#
# Real cross-push resumes (ISSUES.md #29-#31) hit a reproducible failure on
# one specific shard file: the copied local bytes hashed correctly at one
# read and then failed to parse moments later, on the exact same shard,
# across three separate pushes. Retrying the *local* read alone (issues
# #29, #30) never fixed it, because the flaw is in the copy from
# `/kaggle/input`, not in re-reading what was already copied. This cell now
# hash-verifies every shard file immediately after copying it and, on
# mismatch, re-copies fresh bytes from the source mount (not the possibly
# bad local file) up to 3 times before giving up.
prior_manifests = sorted(Path("/kaggle/input").glob("**/full_corpus/shard-manifest.json"))
if prior_manifests:
    prior_dir = prior_manifests[0].parent
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    prior_manifest = json.loads(prior_manifests[0].read_text(encoding="utf-8"))
    expected_hash_by_name = {shard["path"]: shard["sha256"] for shard in prior_manifest.get("shards", [])}
    verified = 0
    for item in prior_dir.iterdir():
        dest = OUT_DIR / item.name
        expected_hash = expected_hash_by_name.get(item.name)
        for copy_attempt in range(3):
            shutil.copy2(item, dest)
            if expected_hash is None:
                break
            actual_hash = hashlib.sha256(dest.read_bytes()).hexdigest()
            if actual_hash == expected_hash:
                verified += 1
                break
            print(f"resume-copy: {item.name} hash mismatch on copy attempt {copy_attempt + 1}/3, re-copying from source")
            if copy_attempt < 2:
                time.sleep(1.0)
        else:
            raise RuntimeError(
                f"resume-copy: {item.name} still fails hash verification after 3 fresh "
                "copies from /kaggle/input; the source mount itself is unreliable for this file"
            )
    shard_count = len(list(OUT_DIR.glob("shard-*.jsonl")))
    print(f"resume: copied and hash-verified {verified} shard file(s) from {prior_dir} ({shard_count} total)")
else:
    print("resume: no prior checkpoint found under /kaggle/input; starting from item 0")


In [ ]:
captioner = Florence2Captioner(revision=CAPTION_MODEL_REVISION)
paraphraser = QwenParaphraser(model_id=PARAPHRASE_MODEL_ID, revision=PARAPHRASE_MODEL_REVISION)

def resolve_image(record, image_dir):
    dest = image_dir / f"{record['global_id']}.bin"
    return download_image(str(record["image_url"]), dest)

# Kaggle enforces a hard ~21,600s (6h) script execution cap (see
# ISSUES.md #22/#24). This budget is measured from KERNEL_START (before
# dependency pinning and metadata streaming), not from this cell, so an
# unexpectedly slow earlier stage automatically leaves less time here
# instead of overrunning the platform's own timeout. 19,800s = 5.5h leaves
# a 30-minute margin for final checkpoint flush and shutdown.
TIME_BUDGET_SECONDS = 19800.0
DEADLINE = KERNEL_START + TIME_BUDGET_SECONDS

IDENTITY = {
    "stage": "full_corpus_generation",
    "catalog_size": CATALOG_SIZE,
    "paraphrase_model": PARAPHRASE_MODEL_ID,
    "paraphrase_revision": PARAPHRASE_MODEL_REVISION,
    "caption_revision": CAPTION_MODEL_REVISION,
    "shuffle_seed": 7001,
    "shard_size": 512,
}

# Conservative first value for T4 VRAM (Florence-2-large + Qwen2.5-3B both
# resident simultaneously); not part of `IDENTITY` since it only affects
# throughput, never the generated captions/paraphrases themselves — safe to
# change between resumed pushes without invalidating the checkpoint.
BATCH_SIZE = 8

records = process_catalog_records_incrementally(
    metadata_records, captioner, paraphraser, OUT_DIR, IDENTITY, IMAGE_DIR, resolve_image,
    shard_size=512, deadline=DEADLINE, batch_size=BATCH_SIZE,
)
print(f"generation complete: {len(records)}/{len(metadata_records)} records")
if len(records) < len(metadata_records):
    print("TIME BUDGET REACHED: add this kernel's own slug to kernel_sources and "
          "re-push the identical notebook to resume from the checkpoint above.")


In [ ]:
import ast
import csv
import json

if len(records) < len(metadata_records):
    print(f"skipping arm assembly: only {len(records)}/{len(metadata_records)} records generated so far; "
          "re-push this identical kernel to resume and finish generation before building arms")
else:
    train_files = sorted((mixed_root / "train").glob("*.csv"))
    sequences = []
    for path in train_files:
        with path.open(encoding="utf-8", newline="") as handle:
            for row in csv.DictReader(handle):
                try:
                    sequences.append([int(value) for value in ast.literal_eval(row["history_item_id"])])
                except (KeyError, SyntaxError, ValueError):
                    continue
    frequencies = compute_training_frequencies(sequences)
    real_cues = {r["global_id"]: r["caption_raw"] for r in records if r.get("caption_status") == "ok"}
    paraphrase_cues = {r["global_id"]: r["paraphrase_text"] for r in records if r.get("paraphrase_status") == "ok"}
    donor_map = frequency_bin_derangement(sorted(real_cues), frequencies, seed=7001)
    rows = [
        CatalogRow(
            domain=str(record["domain"]), source_item_id=str(record["global_id"]),
            downstream_item_id=record["global_id"], parent_asin=record.get("asin"),
            source_title=str(record["title"]), order_index=record["global_id"],
            image_status=str(record.get("image_status", "unknown")),
            caption_status=str(record.get("caption_status", "unknown")),
            raw_caption=record.get("caption_raw"),
        )
        for record in records
    ]
    arms = build_arms(rows, real_cues, paraphrase_cues, donor_map)
    for record, arm in zip(records, arms):
        record["arm_texts"] = arm.texts
        record["shuffle_donor_id"] = arm.shuffle_donor_id

    manifest = write_sharded_records(records, OUT_DIR, IDENTITY, shard_size=512)
    manifest["available_caption_count"] = len(real_cues)
    manifest["image_decoded_count"] = sum(record.get("image_status") == "decoded" for record in records)
    manifest["paraphrase_ok_count"] = sum(record.get("paraphrase_status") == "ok" for record in records)
    (WORK_DIR / "full_corpus_summary.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print(json.dumps({k: v for k, v in manifest.items() if k != "shards"}, indent=2, sort_keys=True))
